# RAG Question Answering with Confluence Documentation

This notebook demonstrates how to:
1. Load the vector database and Confluence JSON data
2. Ask questions about the documentation
3. Retrieve relevant context using vector similarity search
4. Generate answers using the Iliad API
5. Display results with source references

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, HTML

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from config import ConfigConfluenceRag
from rag.vectorstore import VectorStore
from rag.embeddings import EmbeddingManager
from rag.pipeline import RAGPipeline
from loguru import logger

## 1. Validate Configuration

In [ ]:
if not ConfigConfluenceRag.validate():
    raise ValueError("Configuration validation failed. Please check your .env file.")

print("✅ Configuration validated successfully")
print(f"Vector DB Path: {ConfigConfluenceRag.VECTOR_DB_PATH}")
print(f"Embedding Model: {ConfigConfluenceRag.EMBEDDING_MODEL}")
print(f"Iliad API URL: {ConfigConfluenceRag.ILIAD_API_URL}")

## 2. Load Confluence Pages JSON

Load the raw Confluence data for reference and additional context.

In [ ]:
# Load confluence pages JSON
json_path = Path('../Data_Storage/confluence_pages.json')

if not json_path.exists():
    print(f"⚠️  Warning: {json_path} not found.")
    print("Please run 01_data_acquisition.ipynb first to fetch Confluence pages.")
    confluence_pages = []
else:
    with open(json_path, 'r') as f:
        confluence_pages = json.load(f)
    
    print(f"✅ Loaded {len(confluence_pages)} Confluence pages from JSON")
    
    # Display summary statistics
    if confluence_pages:
        df = pd.DataFrame([
            {
                'title': page.get('title', 'Unknown'),
                'space': page.get('space_key', 'Unknown'),
                'author': page.get('author', 'Unknown'),
                'version': page.get('version', 0),
                'text_length': len(page.get('content_text', '')),
            }
            for page in confluence_pages
        ])
        
        print("\nConfluence Pages Summary:")
        display(df.head(10))

## 3. Initialize RAG Pipeline

Set up the vector store, embedding manager, and RAG pipeline with configurable top_k parameter.

In [ ]:
# Initialize vector store
print("Initializing vector store...")
vector_store = VectorStore(
    persist_directory=ConfigConfluenceRag.VECTOR_DB_PATH,
    collection_name='confluence_docs'
)

doc_count = vector_store.count()
print(f"✅ Vector store loaded with {doc_count} document chunks")

if doc_count == 0:
    print("\n⚠️  Warning: Vector store is empty!")
    print("Please run 01_data_acquisition.ipynb first to populate the vector database.")

In [ ]:
# Initialize embedding manager
print("Initializing embedding manager...")
embedding_manager = EmbeddingManager(model_name=ConfigConfluenceRag.EMBEDDING_MODEL)
print(f"✅ Embedding model: {embedding_manager.model_name}")
print(f"   Embedding dimension: {embedding_manager.embedding_dimension}")

In [ ]:
# Initialize RAG pipeline with top_k=10 as default
print("Initializing RAG pipeline...")
pipeline = RAGPipeline(
    vector_store=vector_store,
    embedding_manager=embedding_manager,
    iliad_api_key=ConfigConfluenceRag.ILIAD_API_KEY,
    iliad_api_url=ConfigConfluenceRag.ILIAD_API_URL,
    top_k=10  # Default to 10 similar pages as per requirements
)
print("✅ RAG pipeline initialized successfully")

## 4. Test Query Function

Helper function to ask questions and display formatted results.

In [ ]:
def ask_question(question: str, top_k: int = 10, show_context: bool = False):
    """
    Ask a question using the RAG pipeline and display formatted results.
    
    Args:
        question: The question to ask
        top_k: Number of similar documents to retrieve (default 10)
        show_context: Whether to display the retrieved context chunks
    """
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    try:
        # Query the RAG pipeline
        result = pipeline.query(question, n_results=top_k)
        
        # Display the answer
        display(Markdown("### Answer"))
        display(Markdown(result['answer']))
        
        # Display sources
        if result.get('sources'):
            display(Markdown("\n### Sources"))
            for i, source in enumerate(result['sources'], 1):
                display(Markdown(f"**{i}. {source['title']}**"))
                display(Markdown(f"   - Type: {source['type']}"))
                display(Markdown(f"   - URL: [{source['url']}]({source['url']})"))
        
        # Display relevance scores
        if result.get('distances'):
            display(Markdown("\n### Relevance Scores"))
            scores_df = pd.DataFrame([
                {
                    'Rank': i,
                    'Distance': f"{dist:.4f}",
                    'Similarity': f"{(1-dist)*100:.2f}%"
                }
                for i, dist in enumerate(result['distances'], 1)
            ])
            display(scores_df)
        
        # Optionally show retrieved context
        if show_context and result.get('retrieved_documents'):
            display(Markdown("\n### Retrieved Context Chunks"))
            for i, doc in enumerate(result['retrieved_documents'], 1):
                display(Markdown(f"\n**Chunk {i}:**"))
                display(Markdown(f"```\n{doc[:300]}...\n```"))
        
        return result
        
    except Exception as e:
        print(f"❌ Error processing query: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

## 5. Example Questions

Try asking questions about the Confluence documentation. The RAG system will:
1. Find the top 10 most relevant document chunks from the vector database
2. Pass these chunks along with your question to the Iliad API
3. Generate a contextual answer based on the retrieved information

### Question 1: General Overview

In [ ]:
result1 = ask_question(
    "What data science projects are documented in Confluence?",
    top_k=10
)

### Question 2: Specific Project Information

In [ ]:
result2 = ask_question(
    "What machine learning models are being used in the projects?",
    top_k=10
)

### Question 3: Technical Details

In [ ]:
result3 = ask_question(
    "What technologies and tools are commonly used across the projects?",
    top_k=10
)

### Question 4: Custom Question

Replace with your own question below:

In [ ]:
# Your custom question here
custom_question = "What is the purpose of the customer segmentation project?"

result4 = ask_question(
    custom_question,
    top_k=10,
    show_context=True  # Set to True to see retrieved context chunks
)

## 6. Interactive Question Box

Use this cell to interactively ask questions. Modify the `top_k` parameter to control how many similar documents are retrieved.

In [ ]:
# Interactive question input
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output

# Create widgets
question_input = widgets.Textarea(
    value='',
    placeholder='Enter your question here...',
    description='Question:',
    layout=Layout(width='100%', height='80px')
)

top_k_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description='Top K:',
    continuous_update=False
)

show_context_checkbox = widgets.Checkbox(
    value=False,
    description='Show Context Chunks',
    indent=False
)

submit_button = widgets.Button(
    description='Ask Question',
    button_style='primary',
    icon='search'
)

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        if question_input.value.strip():
            ask_question(
                question_input.value,
                top_k=top_k_slider.value,
                show_context=show_context_checkbox.value
            )
        else:
            print("Please enter a question.")

submit_button.on_click(on_submit_clicked)

# Display widgets
display(widgets.VBox([
    question_input,
    widgets.HBox([top_k_slider, show_context_checkbox]),
    submit_button,
    output_area
]))

## 7. Batch Query Processing

Process multiple questions at once.

In [ ]:
# Define a list of questions to process
batch_questions = [
    "What data sources are commonly used?",
    "What are the key challenges mentioned in the projects?",
    "What is the timeline for the projects?",
]

# Process batch
print("Processing batch questions...\n")
batch_results = []

for i, question in enumerate(batch_questions, 1):
    print(f"\nProcessing question {i}/{len(batch_questions)}...")
    result = ask_question(question, top_k=10)
    if result:
        batch_results.append(result)
    print("\n" + "-"*80)

print(f"\n✅ Completed batch processing: {len(batch_results)} questions answered")

## 8. Export Results

Save query results to a file for later reference.

In [ ]:
import datetime

# Create export directory if it doesn't exist
export_dir = Path('Data_Storage/query_results')
export_dir.mkdir(parents=True, exist_ok=True)

# Save results to JSON
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
export_path = export_dir / f'rag_results_{timestamp}.json'

if batch_results:
    with open(export_path, 'w') as f:
        json.dump(batch_results, f, indent=2)
    print(f"✅ Results exported to: {export_path}")
else:
    print("No results to export. Run some queries first.")

## Summary

This notebook demonstrates the complete RAG question-answering pipeline:
- ✅ Loaded vector database with document chunks
- ✅ Loaded Confluence pages JSON for reference
- ✅ Initialized RAG pipeline with top_k=10 parameter
- ✅ Retrieved relevant context using vector similarity search
- ✅ Generated answers using Iliad API
- ✅ Displayed results with source references and relevance scores
- ✅ Provided interactive and batch query capabilities

The system efficiently answers questions about Confluence documentation by combining retrieval and generation!